# dfx4ml quick start — Part 1 (`config_2`: 1 PR region, 2 RMs)

End-to-end Keras → hls4ml → dfx4ml build for the **2-partition** topology (`halfA`/`halfB` swapped in a single reconfigurable region).

Run from the **repo root**. The model + weight/input lock are built by `build_quick_start_model()` in `example/tutorial/quick_start_hls4ml_model.py`; this notebook covers everything from the topology `CONFIG` through the hardware + software build, and stages the data the board test notebook (`hls4ml_1_region_2_rm.ipynb`) needs.

## 1. Environment + backend
Add `lib/` and the `hls4ml` submodule to the path, register the dfx4ml backend, and import the build helpers.

In [1]:
import os, sys, json, shutil
from pathlib import Path

REPO = Path.cwd()                                # run this notebook from the repo root
sys.path.insert(0, str(REPO / 'lib'))            # lib/hls4ml_build + lib/hls4ml_con
sys.path.insert(0, str(REPO / 'hls4ml'))         # hls4ml submodule source
os.environ['HLS4ML_BACKEND_PLUGINS'] = 'hls4ml_con'   # discovered at `import hls4ml`
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')

import numpy as np
import hls4ml
from hls4ml_build import Hls4ml_build, Partition, Stream, DMA   # orchestrator + typed topology
from lib.hw_build import HwBuildHelper
from lib.sw_build import SwBuildHelper

assert 'vitisunifieddfx4ml' in hls4ml.backends.get_available_backends(), \
    'VitisUnifiedDFx4ml backend not registered — check HLS4ML_BACKEND_PLUGINS / sys.path'
print('OK: VitisUnifiedDFx4ml registered')

OK: VitisUnifiedDFx4ml registered


## 2. Load the model
The Keras model, shared-weight partition sub-models, locked input pool, and probe list come from `build_quick_start_model()` in `example/tutorial/quick_start_hls4ml_model.py`. It is idempotent — the first call trains + locks the weights/input pool to `OUT_ROOT/_lock`; later calls reload them, so results are identical run-to-run.

In [2]:
# the trained full model, the partition sub-models (shared weights), the locked input
# pool and the per-layer probe list all come from
# example/tutorial/quick_start_hls4ml_model.py. build_quick_start_model() is idempotent:
# the first call trains + locks the weights / input pool to OUT_ROOT/_lock; every later
# call (even after a kernel restart, or if _lock was deleted) re-creates / re-loads them,
# so the csim / diag numbers are identical run-to-run.
sys.path.insert(0, str(REPO / 'example' / 'tutorial'))
from quick_start_hls4ml_model import build_quick_start_model

_m = build_quick_start_model()
full_model, halfA, halfB                 = _m.full_model, _m.halfA, _m.halfB
part1, part2, part3, part4               = _m.part1, _m.part2, _m.part3, _m.part4
X_pool, PROBE_LAYERS, OUT_ROOT, LOCK_DIR = _m.X_pool, _m.PROBE_LAYERS, _m.OUT_ROOT, _m.LOCK_DIR
print('model + locked input pool loaded; pool =', X_pool.shape)

full model params: 12532
[train] training 20 epochs to break weight symmetry…
[lock] weights trained + saved → /media/tanawin/tanawin1701e/project8/dfx4ml/dfx4ml_code/hls4ml_dfx_out/_lock/full_weights.h5
partition models: ['full', 'halfA', 'halfB', 'part1', 'part2', 'part3', 'part4']
[lock] saved 1,000 samples → /media/tanawin/tanawin1701e/project8/dfx4ml/dfx4ml_code/hls4ml_dfx_out/_lock/x_input_pool.npy
model + locked input pool loaded; pool = (1000, 8, 8, 1)


## 3. Tool paths
Point at your Vitis / Vivado 2023.2 installs.

In [3]:
# Vitis / Vivado 2023.2 install dirs (each holds settings64.sh); Hls4ml_build.setup_env()
# sources them onto PATH at construction.
VITIS_PATH  = '/tools/Xilinx/Vitis/2023.2'
VIVADO_PATH = '/tools/Xilinx/Vivado/2023.2'

## 4. Topology (`CONFIG`)
The partition cut for this part — one `Partition` per (region, rm).

In [4]:
PART_TAG = 'part_1'      # output root / build_prj / export suffix for this part

# config_2 — single PR region: halfA (rm 0) and halfB (rm 1) are swapped at runtime in
# the same region; all streams live in region 0 (streamers persist across the swap).
CONFIG = [
    Partition('halfA', 'p_halfA', halfA, region=0, rm=0, inputs=[DMA], outputs=[
        Stream('bneck', region=0, alloc_phase=0, free_phase=1),
        Stream('skip2', region=0, alloc_phase=0, free_phase=1),
        Stream('skip1', region=0, alloc_phase=0, free_phase=1)]),
    Partition('halfB', 'p_halfB', halfB, region=0, rm=1,
              inputs=['bneck', 'skip2', 'skip1'], outputs=[DMA]),
]
print('config_2:', [p.name for p in CONFIG])

config_2: ['halfA', 'halfB']


## 5. Run-stage toggles + build config
Flip these to control which stages run. `RUN_DIAG` is optional. `AMT_QUERY` must match the board notebook.

In [5]:
# ── run-stage toggles ──────────────────────────────────────
RUN_CSIM    = True      # end-to-end csim across the partitions (saves x / y / pred)
RUN_DIAG    = False     # OPTIONAL per-layer HLS csim bisect (slow — see the diag cell below)
# C-synthesis stage — pick ONE (mutually exclusive). 'fifo' already runs C-synthesis as
# part of the FIFO-depth optimization, so it supersedes 'synth'; never run both.
#   'none'  — skip synthesis entirely
#   'synth' — C-synthesis + ip_catalog package (needs Vitis)
#   'fifo'  — C-synthesis + FIFO-depth optimization (needs Vitis cosim — heavy)
SYNTH_MODE  = 'fifo'
RUN_HWBUILD = True      # dfx4ml Vivado hardware + PYNQ software build

AMT_QUERY = 900         # csim sample count — must match the board notebook's AMT_QUERY

# per-part output root — keeps part_1 / part_2 artifacts from colliding (the shared
# weight/input lock still lives in OUT_ROOT/_lock; only the build outputs are split).
BUILD_ROOT = OUT_ROOT / PART_TAG

# shared Hls4ml_build construction config (everything except `partitions`)
HB_KW = dict(
    out_root       = BUILD_ROOT,
    board          = 'kv260',
    part           = 'xck26-sfvc784-2LV-c',
    clock_period   = '10ns',
    precision      = 'ap_fixed<16,6>',
    reuse_factor   = 8,
    strategy       = 'Resource',
    total_banks    = 64,
    rm_index_width = 3,
    vitis_path     = VITIS_PATH,
    vivado_path    = VIVADO_PATH,
)

## 6. Construct the orchestrator

In [6]:
hb = Hls4ml_build(partitions=CONFIG, **HB_KW)
print('inferred: amt_phase =', hb.amt_phase, '| num_regions =', hb.num_regions)

v++       -> /tools/Xilinx/Vitis/2023.2/bin/v++
vitis-run -> /tools/Xilinx/Vitis/2023.2/bin/vitis-run
vivado    -> /tools/Xilinx/Vivado/2023.2/bin/vivado
inferred: amt_phase = 1 | num_regions = 1


## 7. Convert (get the partial model)

In [7]:
# 1. get the partial model — convert every partition to an hls4ml ModelGraph + firmware
hb.convert_all()

================================================== convert halfA
================================================== convert halfB
converted: ['halfA', 'halfB']


{'halfA': <hls4ml.model.graph.ModelGraph at 0x76085428ff70>,
 'halfB': <hls4ml.model.graph.ModelGraph at 0x7609080886a0>}

## 8. csim + save reference data
Runs the partitions end-to-end and saves `x_input.npy` / `y_keras.npy` / `y_pred_hls.npy` to `OUT_ROOT/_csim_data` (outside the partition dirs that convert wipes).

In [8]:
# 2. end-to-end csim + save the reference data the board test notebook expects.
# IMPORTANT: write to OUT_ROOT/_csim_data (a sibling of the partition project dirs) —
# convert_all / synth_all(fifo_opt=True) re-create OUT_ROOT/<partition>, so anything saved
# inside a partition folder would be wiped on a re-convert. _csim_data is never touched.
CSIM_DATA_DIR = BUILD_ROOT / '_csim_data'
CSIM_DATA_DIR.mkdir(parents=True, exist_ok=True)

if RUN_CSIM:
    X_csim     = X_pool[:AMT_QUERY].astype(np.float32)                 # x  — board input
    y_keras    = np.asarray(full_model.predict(X_csim), dtype=np.float32)  # y  — float ref
    final, bus = hb.csim_chain(x0=X_csim, peek=10)                     # HLS csim chained out
    y_pred_hls = np.asarray(final, dtype=np.float32)                   # pred — board 'expected'

    np.save(CSIM_DATA_DIR / 'x_input.npy',    X_csim)
    np.save(CSIM_DATA_DIR / 'y_keras.npy',    y_keras)
    np.save(CSIM_DATA_DIR / 'y_pred_hls.npy', y_pred_hls)
    print('saved ->', CSIM_DATA_DIR)
    print('  x_input.npy    ', X_csim.shape)
    print('  y_keras.npy    ', y_keras.shape)
    print('  y_pred_hls.npy ', y_pred_hls.shape)

29/29 [==============================] - 0s 2ms/step
================================================================ halfA
  in  DMA      (900, 8, 8, 1) first10= [0.3745 0.9507 0.7320 0.5987 0.1560 0.1560 0.0581 0.8662 0.6011 0.7081]
  out bneck    (900, 32)      first10= [0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000]
  out skip2    (900, 256)     first10= [0.0117 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0107 0.0078]
  out skip1    (900, 512)     first10= [0.0000 0.0000 0.0000 0.0000 0.3076 0.0000 0.0000 0.4561 0.0303 0.0898]
================================================================ halfB
  in  bneck    (900, 32)      first10= [0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000]
  in  skip2    (900, 256)     first10= [0.0117 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0107 0.0078]
  in  skip1    (900, 512)     first10= [0.0000 0.0000 0.0000 0.0000 0.3076 0.0000 0.0000 0.4561 0.0303 0.0898]
  out DMA      (900, 4)     

## 9. (optional) per-layer diagnostic
Set `RUN_DIAG=True` for a per-layer fixed-point signal-collapse bisect. Safe to skip.

In [9]:
# OPTIONAL — leave RUN_DIAG=False unless you want the per-layer fixed-point
# signal-collapse report. diag is split-independent (it probes the full model), so it is
# identical for both parts and is run on a tiny slice (it is slow).
if RUN_DIAG:
    hb.diag_bisect(full_model, PROBE_LAYERS, X_pool[:2])

## 10. Streamer glue

In [10]:
# 4. streamer glue — compute dfx params (-> hb.dfx) and stitch the user-BD dispatcher TCL
hb.compute_streamer_glue()
hb.print_streamer_report()

[dfx-streamer] streams:
  ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  | name       | shape              | region | alloc_phase | free_phase | precision | amt_entry_per_query | bits_per_entry | amt_banks_per_entry | amt_query_per_bankGrp |
  ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  | bneck      | (2, 2, 8)          |      0 |           0 |          1 |        16 |                   4 |            128 |                   2 |                  1024 |
  | skip2      | (4, 4, 16)         |      0 |           0 |          1 |        16 |                  16 |            256 |                   4 |                   256 |
  | skip1      | (8, 8, 8)          |      0 |           0 |          1 |        16 |                  64 |            128 |                   2 |               

## 11. Synthesis + FIFO optimization

In [11]:
# 3. synthesis + package (needs Vitis). Driven by the single SYNTH_MODE selector —
# 'synth' and 'fifo' are mutually exclusive ('fifo' = synth + FIFO-depth optimization,
# so it already runs C-synthesis; we never run plain synth on top of it).
# FIFO-depth optimization re-converts each partition (wiping + rewriting
# OUT_ROOT/<partition>) — the _csim_data files survive.
assert SYNTH_MODE in ('none', 'synth', 'fifo'), f"SYNTH_MODE must be 'none'|'synth'|'fifo', got {SYNTH_MODE!r}"
if SYNTH_MODE == 'synth':
    hb.synth_all()
elif SYNTH_MODE == 'fifo':
    hb.synth_all(fifo_opt=True)
else:
    print('SYNTH_MODE = none — skipping synthesis')

================================================== fifo-opt halfA
FIFO optimization completed

****** v++ v2023.2 (64-bit)
  **** SW Build 4026344 on 2023-10-11-15:42:10
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2023 Advanced Micro Devices, Inc. All Rights Reserved.

Running Dispatch Server on port: 43499
INFO: [v++ 60-1548] Creating build summary session with primary output /media/tanawin/tanawin1701e/project8/dfx4ml/dfx4ml_code/hls4ml_dfx_out/part_1/halfA/vitis_workspace/p_halfA/vitis_unified_project/vitis_unified_project.hlscompile_summary, at Fri Jun 19 16:40:29 2026
INFO: [v++ 82-31] Launching vitis_hls: vitis_hls -nolog -run csynth -work_dir /media/tanawin/tanawin1701e/project8/dfx4ml/dfx4ml_code/hls4ml_dfx_out/part_1/halfA/vitis_workspace/p_halfA/vitis_unified_project -config /media/tanawin/tanawin1701e/project8/dfx4ml/dfx4ml_code/hls4ml_dfx_out/part_1/halfA/hls_kernel_config.cfg -cmdlineconfig /media/tanawin/tanawin1701e/project8/dfx4ml/

## 12. Hardware build

In [12]:
# 5. dfx4ml Vivado hardware build (board / user_repo / tcl / dfx / rm_width / vivado
# are all pulled from hb).
hw = None
if RUN_HWBUILD:
    hw = HwBuildHelper(
        build_folder_path  = f'./build_prj_{PART_TAG}',
        dfx_root_path      = '.',
        export_folder_path = f'./export_{PART_TAG}',
        req_gen_ip         = 1,
        num_core           = 4,
        clk_frq            = 99999001,
        test_mode          = 0,          # user kernels
        hls4ml_build       = hb,         # config pulled from hb
    )
    hw.run_build()
    hw.package_export_files()
    print('hardware build complete -> ./export/hw')

Running Vivado with /media/tanawin/tanawin1701e/project8/dfx4ml/dfx4ml_code/build_prj_part_1/run_build.tcl...

****** Vivado v2023.2 (64-bit)
  **** SW Build 4029153 on Fri Oct 13 20:13:54 MDT 2023
  **** IP Build 4028589 on Sat Oct 14 00:45:43 MDT 2023
  **** SharedData Build 4025554 on Tue Oct 10 17:18:54 MDT 2023
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2023 Advanced Micro Devices, Inc. All Rights Reserved.

start_gui
INFO: [Common 17-206] Exiting Vivado at Fri Jun 19 19:27:26 2026...





hardware build complete -> ./export/hw


## 13. Software build

In [13]:
# 6. PYNQ software build — packages the drivers + data folder into ./export
if RUN_HWBUILD:
    SwBuildHelper(hw_builder=hw).package_export_file()
    print('software build complete -> ./export')

software build complete -> ./export


## 14. Stage data + board notebook into ./export
Copies the csim reference arrays into `export/data/` and drops the matching board-side test notebook beside `export/`.

In [14]:
# 7. stage the csim reference data + the matching board-side test notebook into ./export.
# On the KV260 the export folder is the project dir: the board notebook loads
# data/x_input.npy and compares the hardware output to data/y_pred_hls.npy.
EXPORT_DIR  = REPO / f'export_{PART_TAG}'
EXPORT_DATA = EXPORT_DIR / 'data'
EXPORT_DATA.mkdir(parents=True, exist_ok=True)

for fname in ('x_input.npy', 'y_keras.npy', 'y_pred_hls.npy'):
    src = CSIM_DATA_DIR / fname
    if src.exists():
        shutil.copy(src, EXPORT_DATA / fname)
        print('copied', fname, '->', EXPORT_DATA / fname)

BOARD_NB = REPO / 'example' / 'tutorial' / 'hls4ml_1_region_2_rm.ipynb'
shutil.copy(BOARD_NB, EXPORT_DIR / BOARD_NB.name)
print('copied board notebook ->', EXPORT_DIR / BOARD_NB.name)

copied x_input.npy -> /media/tanawin/tanawin1701e/project8/dfx4ml/dfx4ml_code/export_part_1/data/x_input.npy
copied y_keras.npy -> /media/tanawin/tanawin1701e/project8/dfx4ml/dfx4ml_code/export_part_1/data/y_keras.npy
copied y_pred_hls.npy -> /media/tanawin/tanawin1701e/project8/dfx4ml/dfx4ml_code/export_part_1/data/y_pred_hls.npy
copied board notebook -> /media/tanawin/tanawin1701e/project8/dfx4ml/dfx4ml_code/export_part_1/hls4ml_1_region_2_rm.ipynb
